In [1]:
import networkx as nx
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd

In [73]:
## Import data from text file and import list of "essential" proteins

raw = nx.read_weighted_edgelist("protein.links.clean.txt",comments="#",nodetype=str)
raw_phys = nx.read_weighted_edgelist("protein.physical.links.clean.txt",comments="#",nodetype=str)

essential = pd.read_csv("Data - List of essential proteins of saccharomyces cerevisiae.csv", header=None)
ess_prot = set(essential.iloc[:,1])

ess_graph = raw.subgraph(ess_prot).copy()
## ----------------------------------------------------------------

In [76]:
target = "YMR161W"
confidence = 750  ## Minimum edge strength (anything below ignored)

graph = raw.copy()
physical = raw_phys.copy()

# removes any edges below the confidence score
for e in graph.edges:
    weight = list(graph.get_edge_data(e[0],e[1]).values())
    if(weight[0] <= confidence):
        graph.remove_edge(e[0],e[1])

# Identifies largest connected component and ignores any disconnected nodes
largest = max(nx.connected_components(graph), key=len)
graph = graph.subgraph(largest).copy()


## Create a copy with the target node removed for analysis
if target in graph:
    edit = graph.copy()
    edit.remove_node(target)
else:
    print(f"{target} not included in network") 


# Repeat for Physical links data -------------------------------------------------------------------
# --------------------------------------------------------------------------------------------------

# removes any edges below the confidence score
for e in physical.edges:
    weight = list(physical.get_edge_data(e[0],e[1]).values())
    if(weight[0] <= confidence):
        physical.remove_edge(e[0],e[1])

# Identifies largest connected component and ignores any disconnected nodes
largest = max(nx.connected_components(physical), key=len)
physical = physical.subgraph(largest).copy()

## Create a copy with the target node removed for analysis
if target in physical:
    phys_edit = physical.copy()
    phys_edit.remove_node(target)
else:
    print(f"{target} not included in physical network") 

YMR161W not included in physical network


In [64]:
## Create set of nodes local to target node

# note distance of 4: 4586, 3: 2437, 2: 254, ___ (5519 total)
distance = 1

local_nodes = set(nx.single_source_shortest_path_length(graph, target, cutoff=distance).keys())


print(len(local_nodes))
print(len(graph))

20
5519


In [69]:
print(nx.shortest_path(graph, "YMR161W", "YJL162C"))
print(local_nodes)

['YMR161W', 'YFR041C', 'YJL162C']
{'YBL075C', 'YNL328C', 'YHR064C', 'YLR008C', 'YGL128C', 'YMR186W', 'YLL024C', 'YMR161W', 'YEL030W', 'YAL005C', 'YER103W', 'YPR061C', 'YGR285C', 'YJL034W', 'YJR097W', 'YKL073W', 'YFR041C', 'YGL018C', 'YDR320C', 'YPL106C'}


In [77]:
# Compare with and without HLJ1

control = nx.pagerank(graph)
measure = nx.pagerank(edit)

difference = {node: control[node] - measure[node] for node in edit.nodes}

relative = {node: difference[node] / measure[node] for node in edit.nodes}

top_20 = sorted(relative.items(), key=lambda x: -abs(x[1]))[:20]

not_local = [(node, diff) for node, diff in top_20 if node not in local_nodes]

for node, diff in not_local:
    print(f"{node}: {diff:+.5f}")

YJL162C: -0.00396
